# 원본 DINOv2 모델: 재활용 사진 판정 점검

팀원의 `colab_fast_model_v2.ipynb` 사본이다. **원본 전처리·특징·로지스틱 회귀·5-fold 분할을 유지**한다.
추가한 것은 입력 검증, 특징의 중간 저장, fold 모델 보관, 재활용 목록별 결과 집계다.

- A: 원본 크롭 / B: 원본 코드의 마스킹 / C: 원본 게이트 / D: 원본 게이트+마스킹.
- 여기서 원본 마스킹은 **원본 DINOv2에 포함된 질감 v2**다. 팀원 B 마스킹이나 형상 v2 마스크로 교체하지 않는다.
- 재활용 **743장(정상 379, 언더필 364)**은 결과를 골라 보는 평가 목록이다. 형상 v2 면적 10% 미만 61장은 이 목록에서만 제외한다.
- 원본 학습 대상은 그대로: A/B는 전체 4,000장, C/D는 원본 게이트 통과 사진을 각 fold의 train에서 학습한다.
- 재활용 사진도 해당 사진을 학습하지 않은 fold 모델(OOF)로 평가한다. 최종 전체학습 모델로 성능을 재지 않는다.
- 4개 조건의 공통 판정 사진 점수와, 743장 전체에서의 판정/보류 비율을 함께 저장한다. 보류는 정상이 아니다.

**준비:** Drive `산공종설` 폴더에 원본 `underfill_originals_4000.zip`과 함께 제공한 목록 ZIP을 업로드한다.
Colab에서 **GPU 런타임**을 고른 뒤 위에서부터 실행한다. 이 PC에서는 본 학습을 실행하지 않았다.

## 1. 실행 경로와 환경
처음 실행은 pretrained 가중치 다운로드와 4,000장×2입력 특징 추출이 필요하다.
로지스틱 회귀는 가볍지만 특징 추출까지 몇 초 만에 끝난다는 뜻은 아니다. 첫 배치 이후 ETA를 확인한다.
중간 특징과 완료 행 수를 Drive에 저장한다. 끊긴 뒤 같은 OUT으로 다시 실행하면 완료한 배치를 재사용한다.
이 폴더에서 동시에 두 런타임을 실행하지 않는다. 새 실험이면 RUN_NAME을 바꾼다.

In [ ]:
import os, sys, json, time, hashlib, zipfile, shutil, subprocess, platform
from pathlib import Path
import torch
assert torch.cuda.is_available(), '런타임 유형을 GPU로 변경해 줘.'
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/산공종설'
RUN_NAME = 'dinov2_original_rescue_20261005_run01'
OUT = f'{DRIVE_DIR}/results/{RUN_NAME}'
LOCAL = Path('/content/dinov2_original_check')
LOCAL.mkdir(exist_ok=True)
Path(OUT).mkdir(parents=True, exist_ok=True)
ORIGINAL_ZIP = Path(DRIVE_DIR) / 'underfill_originals_4000.zip'
SUPPORT_ZIP = Path(DRIVE_DIR) / 'dinov2_original_rescue_support_20261005.zip'
assert ORIGINAL_ZIP.is_file() and SUPPORT_ZIP.is_file(), 'Drive에 원본 ZIP과 목록 ZIP을 업로드해 줘.'

def safe_extract(zpath, target):
    target = Path(target).resolve()
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for n in z.namelist():
            p = (target / n).resolve()
            assert p.is_relative_to(target), f'잘못된 ZIP 경로: {n}'
        z.extractall(target)

SUPPORT = LOCAL / 'support'
safe_extract(SUPPORT_ZIP, SUPPORT)
def file_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''):
            h.update(b)
    return h.hexdigest()
checksums = json.loads((SUPPORT / 'checksums.json').read_text())
for name, expected in checksums.items():
    assert file_sha(SUPPORT / name) == expected, f'목록 ZIP 파일 손상: {name}'
FOLDS = str(SUPPORT / 'folds_all4000.csv')
with open(f'{OUT}/package_install.log', 'w') as log:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'scikit-learn'],
                   stdout=log, stderr=subprocess.STDOUT, check=True)
import timm, torchvision, sklearn, numpy, pandas, cv2
ENV = dict(python=platform.python_version(), torch=torch.__version__,
           torchvision=torchvision.__version__, timm=timm.__version__, sklearn=sklearn.__version__,
           numpy=numpy.__version__, pandas=pandas.__version__, opencv=cv2.__version__,
           cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0))
print(ENV)
Path(f'{OUT}/environment.json').write_text(json.dumps(ENV, indent=2), encoding='utf-8')

## 2. 게이트 + 마스킹 코드 (filament_gate.py v2)

In [ ]:
%%writefile /content/dinov2_original_check/filament_gate.py
"""질감 기준 필라멘트 게이트 + 베드 마스킹 v2 (2026-10-02, 팀원)

하는 일 (고정 크롭 1000x1000 하나에 대해)
  1. 출력물 영역 찾기 -> 면적 비율(ratio)
  2. 게이트: ratio < 0.1 이면 판정 보류 (베드만 찍힌 사진)
  3. 마스킹: 출력물 바깥(베드)을 회색으로 칠한 크롭을 돌려줌 -> 모델 입력용
     (모델이 '반짝이 = 언더필'로 판단하는 문제를 막기 위함)

출력물 영역 찾는 순서 (250x250으로 줄여서 계산)
  a. 픽셀마다 주변 9x9 밝기 표준편차(질감)와 평균 밝기 계산
  b. 표준편차 < 10 (매끈) & 밝기 > 110 -> 출력물 후보
     (v1의 채도 조건은 삭제: 360장 시험에서 빼도 결과 동일, 자동으로 정하면 흰 출력물끼리 갈라버림)
  c. 7x7 열기·닫기, 화면 2% 미만 덩어리 버림
  d. 흔들린 베드 얼룩 제거: 덩어리의 (표준편차 중앙값 >= 5) & (밝기 중앙값 < 155) & (볼록도 < 0.8)
     -> 셋 다 해당하면 버림. 번진 반짝이는 덜 매끈하고 어둡고 모양이 들쭉날쭉함
  e. 남은 덩어리마다 바깥 윤곽 안쪽을 전부 채움 -> 언더필의 틈(결함 증거)이 회색으로 지워지지 않게
  f. 마스킹 때는 윤곽을 2픽셀(원본 기준 약 8픽셀) 넓혀서 테두리가 잘리지 않게 함

사용법
  from filament_gate import crop_roi, filament_ratio, gate, mask_bed
  crop = crop_roi(img)                     # 원본 1600x1200 -> 고정 크롭 1000x1000
  r = filament_ratio(crop)                 # RGB 기본 (본인 pipeline.py와 같은 형태)
  ok, r = gate(crop, bgr=True)             # cv2로 읽은 BGR이면 bgr=True. ok=False면 판정 보류
  masked, r = mask_bed(crop, bgr=True)     # 베드를 회색 칠한 크롭 (입력과 같은 크기·색 순서)

필요 패키지: numpy, opencv-python (Colab 기본 설치됨)
"""
import cv2
import numpy as np

VERSION = "v2-mask"
ROI = (0, 250, 1000, 1250)       # ymin, xmin, ymax, xmax
THRESHOLD = 0.1                  # 출력물 면적 이 비율 미만이면 판정 보류
SIZE, WIN = 250, 9               # 계산용 축소 크기, 질감 보는 창
SD_T, MEAN_T = 10, 110           # 출력물 후보: 매끈 & 밝음
MIN_BLOB = 0.02                  # 화면 2% 미만 덩어리 버림
BLUR_SD, BLUR_MEAN, BLUR_SOL = 5, 155, 0.8   # 흔들린 베드 얼룩 판정 (셋 다 해당하면 버림)
MARGIN = 2                       # 마스킹 여유 (축소 기준 픽셀)
GRAY_RGB = (124, 116, 104)       # ImageNet 평균색 -> 정규화하면 거의 0
_K7 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
_KM = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * MARGIN + 1, 2 * MARGIN + 1))


def crop_roi(img):
    """원본 프레임 -> 고정 크롭. 색 순서 상관없음."""
    y0, x0, y1, x1 = ROI
    assert img.shape[0] >= y1 and img.shape[1] >= x1, f"원본 크기가 크롭보다 작음: {img.shape}"
    return img[y0:y1, x0:x1]


def object_mask(crop, bgr=False):
    """출력물 영역 마스크 (SIZE x SIZE, 0/1, 윤곽 안 채움)."""
    crop = np.ascontiguousarray(crop, dtype=np.uint8)
    small = cv2.resize(crop, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY if bgr else cv2.COLOR_RGB2GRAY).astype(np.float32)
    m = cv2.blur(g, (WIN, WIN))
    sd = np.sqrt(np.maximum(cv2.blur(g * g, (WIN, WIN)) - m * m, 0))
    raw = ((sd < SD_T) & (m > MEAN_T)).astype(np.uint8)
    raw = cv2.morphologyEx(cv2.morphologyEx(raw, cv2.MORPH_OPEN, _K7), cv2.MORPH_CLOSE, _K7)
    n, lab, st, _ = cv2.connectedComponentsWithStats(raw)
    out = np.zeros_like(raw)
    for j in range(1, n):
        if st[j, cv2.CC_STAT_AREA] < MIN_BLOB * SIZE * SIZE:
            continue
        comp = lab == j
        cs, _ = cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        c = max(cs, key=cv2.contourArea)
        sol = cv2.contourArea(c) / max(cv2.contourArea(cv2.convexHull(c)), 1.0)
        if np.median(sd[comp]) >= BLUR_SD and np.median(m[comp]) < BLUR_MEAN and sol < BLUR_SOL:
            continue                                    # 흔들린 베드 얼룩
        cv2.drawContours(out, cs, -1, 1, thickness=-1)  # 바깥 윤곽 안쪽 전부 채움 (틈 유지)
    return out


def filament_ratio(crop, bgr=False):
    """출력물 면적 비율 (0~1). 본인 filament_ratio(rgb_crop)와 같은 형태."""
    return float(object_mask(crop, bgr).mean())


def gate(crop, threshold=THRESHOLD, bgr=False):
    """(통과 여부, 비율). 통과 못 하면 판정 보류."""
    r = filament_ratio(crop, bgr)
    return r >= threshold, r


def mask_bed(crop, bgr=False):
    """베드(출력물 바깥)를 회색으로 칠한 크롭과 비율. 크기·색 순서는 입력과 같음."""
    crop = np.ascontiguousarray(crop, dtype=np.uint8)
    om = object_mask(crop, bgr)
    keep = cv2.dilate(om, _KM)
    keep = cv2.resize(keep, (crop.shape[1], crop.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
    out = crop.copy()
    out[~keep] = GRAY_RGB[::-1] if bgr else GRAY_RGB
    return out, float(om.mean())


def overlay(crop, bgr=False, color=(255, 40, 40), alpha=0.55):
    """확인용: 출력물로 판단한 영역을 빨간색으로 칠한 RGB 이미지 (SIZE x SIZE)."""
    om = object_mask(crop, bgr)
    small = cv2.resize(np.ascontiguousarray(crop, dtype=np.uint8), (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    rgb = small[:, :, ::-1].copy() if bgr else small.copy()
    k = om == 1
    rgb[k] = ((1 - alpha) * rgb[k] + alpha * np.array(color)).astype(np.uint8)
    return rgb


def otsu_ratio(crop, bgr=False):
    """비교용: 기존 게이트 값 (팀원 A check_filament_presence와 같은 Otsu 밝은 픽셀 비율)."""
    g = cv2.cvtColor(np.ascontiguousarray(crop, dtype=np.uint8), cv2.COLOR_BGR2GRAY if bgr else cv2.COLOR_RGB2GRAY)
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return float((b > 0).mean())


def _self_check():
    flat = np.full((1000, 1000, 3), 200, np.uint8)
    rng = np.random.default_rng(0)
    glitter = (rng.random((1000, 1000, 3)) > 0.7).astype(np.uint8) * 230
    assert filament_ratio(flat) > 0.95, filament_ratio(flat)
    assert filament_ratio(glitter) < 0.05, filament_ratio(glitter)
    half = glitter.copy(); half[500:] = 200
    mk, r = mask_bed(half)
    assert 0.4 < r < 0.6 and (mk[:400] == GRAY_RGB).all() and (mk[600:] == 200).all()


if __name__ == "__main__":
    # python filament_gate.py 사진.jpg ... -> 사진별 비율, 통과/보류, 기존 Otsu 값
    import sys
    _self_check()
    for p in sys.argv[1:]:
        img = cv2.imread(p)
        ok, r = gate(crop_roi(img), bgr=True)
        print(f"{p}\tratio={r:.3f}\t{'통과' if ok else '보류'}\t(otsu={otsu_ratio(crop_roi(img), bgr=True):.3f})")
    if len(sys.argv) == 1:
        print("self-check ok", VERSION)

## 3. 원본 풀기 + 게이트 불러오기
마지막 줄에 `4000`과 `v2-mask`가 나와야 합니다.

In [ ]:
import importlib
ORIG = LOCAL / 'orig'
MARKER = LOCAL / 'original_extract.json'
zip_stamp = dict(path=str(ORIGINAL_ZIP), size=ORIGINAL_ZIP.stat().st_size,
                 mtime=ORIGINAL_ZIP.stat().st_mtime_ns)
if MARKER.exists():
    assert json.loads(MARKER.read_text()) == zip_stamp, '원본 ZIP이 바뀜: 새 런타임에서 실행해 줘.'
else:
    safe_extract(ORIGINAL_ZIP, ORIG)
    MARKER.write_text(json.dumps(zip_stamp))
sys.path.insert(0, str(LOCAL))
sys.path.insert(0, str(SUPPORT))
import filament_gate as fg
importlib.reload(fg)
fg._self_check()
print('원본 게이트:', fg.VERSION, fg.ROI, '원본 추출:', len(list(ORIG.rglob('*.jpg'))))

## 4. 설정 + 사진 목록

In [ ]:
import numpy as np, pandas as pd, cv2, time
from pathlib import Path
GATE_T, PROB_T, IMG = fg.THRESHOLD, 0.5, 518
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
folds = pd.read_csv(FOLDS, encoding="utf-8-sig")
folds["source"] = folds.source.str.replace("\\", "/", regex=False)
imgs = (folds[folds.role == "val"][["source", "label", "label_id", "frame_index", "fold"]]
        .rename(columns={"fold": "test_fold"}).sort_values("source").reset_index(drop=True))
src_idx = {s: i for i, s in enumerate(imgs.source)}
print(len(imgs), imgs.label.value_counts().to_dict())

# 파일명으로 위치를 추정해 재라벨링하지 않는다. 기존 공통 분할을 검증한다.
manifest = pd.read_csv(SUPPORT / 'source_manifest.csv', encoding='utf-8-sig')
assert len(imgs) == imgs.source.nunique() == 4000
assert len(folds) == 20000 and set(folds.fold) == set(range(5))
assert imgs.label.value_counts().to_dict() == {'normal': 2000, 'underfill_50': 2000}
assert set(imgs.source) == set(manifest.source)
assert imgs.set_index('source').label_id.equals(manifest.set_index('source').label_id.reindex(imgs.source))
for k in range(5):
    f = folds[folds.fold == k]
    assert len(f) == 4000 and f.source.nunique() == 4000
    for label, first in [('normal', 2177), ('underfill_50', 1281)]:
        a = f[f.label == label]
        low, high = first + k*400, first + (k+1)*400-1
        expected_role = np.where(a.frame_index.between(low, high), 'val',
                        np.where(a.frame_index.between(low-30, high+30), 'buffer', 'train'))
        assert np.array_equal(a.role.to_numpy(), expected_role), '시간 블록/30프레임 buffer 불일치'
source_checksums = dict(zip(manifest.source, manifest.source_sha256))
bad = [s for s in imgs.source if not (ORIG / s).is_file()]
assert not bad, f'원본 폴더 구조가 예상과 다름: {bad[:3]}'
print('검증 완료: 5-fold, 클래스별 400장 검증 블록, 경계 buffer 30프레임.')

## 5. 원본 확인과 원본 게이트 값
이번 실행 폴더의 캐시만 사용한다. 다른 실험의 CSV를 자동으로 가져오지 않는다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
# 먼저 4,000장 전부의 파일 해시를 검증한다. 다른 원본과 캐시가 섞이지 않게 한다.
t0 = time.time()
def check_source(s):
    assert file_sha(ORIG / s) == source_checksums[s], f'원본 파일 불일치: {s}'
with ThreadPoolExecutor(4) as ex:
    list(ex.map(check_source, imgs.source))
GATE_SHA = file_sha(LOCAL / 'filament_gate.py')
data_settings = dict(checksums=checksums, gate_sha256=GATE_SHA,
                     sources=imgs.source.tolist(), threshold=GATE_T, rounding=4)
settings_path = Path(OUT) / 'data_settings.json'
if settings_path.exists():
    assert json.loads(settings_path.read_text()) == data_settings, '기존 OUT의 입력/분할/게이트가 다름: RUN_NAME 변경 필요'
else:
    settings_path.write_text(json.dumps(data_settings, indent=2))
gate_file = f'{OUT}/gate_values_v2.csv'
if os.path.exists(gate_file):
    gv = pd.read_csv(gate_file, encoding='utf-8-sig')
else:
    def one(s):
        image = cv2.imread(str(ORIG / s))
        assert image is not None and image.shape == (1200, 1600, 3), f'이미지 오류: {s}'
        return fg.filament_ratio(fg.crop_roi(image), bgr=True)
    with ThreadPoolExecutor(4) as ex:
        vals = list(ex.map(one, imgs.source))
    gv = pd.DataFrame({'source': imgs.source, 'new': np.round(vals, 4)})
    gv.to_csv(gate_file, index=False, encoding='utf-8-sig')
assert len(gv) == gv.source.nunique() == 4000 and set(gv.source) == set(imgs.source)
imgs = imgs.drop(columns=['new'], errors='ignore').merge(gv[['source', 'new']], on='source', how='left')
assert imgs.source.tolist() == data_settings['sources'] and imgs.new.notna().all()
imgs['passed'] = imgs.new >= GATE_T
print('원본 게이트 통과:', imgs.groupby('label').passed.agg(['sum', 'count']).to_string())
print(f'입력 검증/게이트 {time.time()-t0:.0f}초')

## 6. DINOv2 특징 추출: 마스킹 전·후 두 벌 (한 번만)
사진마다 원본 크롭과 마스킹 크롭을 둘 다 넣어 `feats_nomask.npy`, `feats_mask.npy`로 저장합니다. 이미 있으면 불러오기만 합니다. 처음 실행 때 DINOv2 가중치를 받습니다(HF_TOKEN 경고는 무시). 걸린 시간이 마지막에 출력됩니다.

In [ ]:
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.transforms.v2 import functional as TF

def to_input(crop_rgb):
    # RGB uint8 크롭 -> 518 축소 -> 0~1 -> ImageNet 정규화
    t = torch.from_numpy(np.ascontiguousarray(crop_rgb)).permute(2, 0, 1)
    t = TF.resize(t, [IMG, IMG], interpolation=v2.InterpolationMode.BILINEAR, antialias=True)
    return TF.normalize(t.float() / 255.0, MEAN, STD)

def both_inputs(bgr):
    crop = fg.crop_roi(bgr)
    masked, _ = fg.mask_bed(crop, bgr=True)
    return to_input(crop[:, :, ::-1]), to_input(masked[:, :, ::-1])

dev = "cuda"
dn = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0).eval().to(dev)
NPT = dn.num_prefix_tokens

@torch.no_grad()
def extract(x):
    with torch.autocast("cuda", dtype=torch.float16):
        tok = dn.forward_features(x.to(dev)).float()
    return torch.cat([tok[:, 0], tok[:, NPT:].mean(1)], 1).cpu()   # CLS + 조각 평균



# 입력 변환/특징은 원본과 같다. 저장 방법만 중간 재개 가능한 memmap으로 바꾼다.
dn.requires_grad_(False)
wh = hashlib.sha256()
for key, value in dn.state_dict().items():
    wh.update(key.encode())
    wh.update(value.detach().cpu().contiguous().numpy().tobytes())
feature_settings = dict(data=data_settings, weights_sha256=wh.hexdigest(), env=ENV,
                        model='vit_base_patch14_dinov2.lvd142m', img=IMG, mean=MEAN, std=STD,
                        feature='CLS+mean_patch', dtype='float32', autocast='float16', dim=1536,
                        resize='uint8 tensor bilinear antialias', augmentation=False)
metadata_path = Path(OUT) / 'feature_settings.json'
if metadata_path.exists():
    assert json.loads(metadata_path.read_text()) == feature_settings, '캐시 환경/가중치 불일치: RUN_NAME 변경 필요'
else:
    metadata_path.write_text(json.dumps(feature_settings, indent=2))
ff = {k: f'{OUT}/feats_{k}.npy' for k in ['nomask', 'mask']}
progress_path = Path(OUT) / 'feature_progress.json'
progress = json.loads(progress_path.read_text()) if progress_path.exists() else {'completed_rows': 0}
start = int(progress['completed_rows'])
assert 0 <= start <= len(imgs)
X = {}
for k, path in ff.items():
    if os.path.exists(path):
        X[k] = np.lib.format.open_memmap(path, mode='r+')
        assert X[k].shape == (len(imgs), 1536) and X[k].dtype == np.float32
    else:
        assert start == 0, '완료 표시와 캐시 파일이 맞지 않음: RUN_NAME 변경 필요'
        X[k] = np.lib.format.open_memmap(path, mode='w+', dtype=np.float32, shape=(len(imgs), 1536))
class Imgs(Dataset):
    def __len__(self): return len(imgs)-start
    def __getitem__(self, i):
        image = cv2.imread(str(ORIG / imgs.source[start+i]))
        assert image is not None
        return both_inputs(image)
BATCH_SIZE = 16  # OOM이면 8로 낮춰 같은 OUT에서 재실행. 학습 규칙은 바뀌지 않는다.
t0, cursor = time.time(), start
loader = DataLoader(Imgs(), batch_size=BATCH_SIZE, num_workers=2, shuffle=False)
with open(f'{OUT}/feature_progress.log', 'a', encoding='utf-8') as log:
    for batch, (x0, x1) in enumerate(loader):
        f0, f1 = extract(x0).numpy(), extract(x1).numpy()
        assert f0.shape[1] == f1.shape[1] == 1536 and np.isfinite(f0).all() and np.isfinite(f1).all()
        stop = cursor + len(f0)
        X['nomask'][cursor:stop], X['mask'][cursor:stop] = f0, f1
        for v in X.values(): v.flush()
        # 완료 행 수는 두 입력의 저장 이후에만 기록한다. 불완전 배치는 다시 계산한다.
        temp = progress_path.with_suffix('.tmp')
        temp.write_text(json.dumps({'completed_rows': stop, 'total_rows': len(imgs)}))
        os.replace(temp, progress_path)
        cursor = stop
        if batch % 10 == 0 or cursor == len(imgs):
            elapsed = time.time()-t0
            eta = elapsed/(cursor-start)*(len(imgs)-cursor)
            message = f'{cursor}/{len(imgs)} | 경과 {elapsed/60:.1f}분 | 남음 추정 {eta/60:.1f}분'
            print(message); log.write(message+'\n'); log.flush()
assert int(json.loads(progress_path.read_text())['completed_rows']) == len(imgs)
print('특징 완료:', {k: v.shape for k, v in X.items()})

## 7. 원본의 4조건 × 5-fold 학습
특징 추출기는 고정하고 분류층인 로지스틱 회귀만 학습한다.
표준화는 각 fold 학습 사진에만 fit한다. 원본의 C=1, max_iter=5000, 기본 lbfgs를 유지한다.
원본에 별도 시드가 지정돼 있지 않아 임의 시드를 추가하지 않는다(증강 없음).
조건별 판정 사진이 다르므로 전체 정확도로만 순위를 매기지 않는다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def fit(Xtr, ytr):
    sc = StandardScaler().fit(Xtr)
    return sc, LogisticRegression(max_iter=5000).fit(sc.transform(Xtr), ytr)

def cv_prob(feat, use_gate, condition):
    p = np.full(len(imgs), np.nan); Xf = X[feat]; y = imgs.label_id.values
    for k in range(5):
        f = folds[folds.fold == k]
        itr, ite = f[f.role == "train"].source.map(src_idx).values, f[f.role == "val"].source.map(src_idx).values
        if use_gate:
            itr, ite = itr[imgs.passed.values[itr]], ite[imgs.passed.values[ite]]
        sc, clf = fit(Xf[itr], y[itr])
        p[ite] = clf.predict_proba(sc.transform(Xf[ite]))[:, 1]
        import joblib
        model_dir = Path(OUT) / 'fold_models' / condition[0]
        model_dir.mkdir(parents=True, exist_ok=True)
        joblib.dump({'scaler': sc, 'clf': clf, 'fold': k, 'condition': condition,
                     'train_sources': imgs.source.iloc[itr].tolist(),
                     'val_sources': imgs.source.iloc[ite].tolist()}, model_dir / f'fold{k}.joblib')
        message = f'{condition} fold{k}: train={len(itr)}, val={len(ite)}'
        print(message)
        with open(f'{OUT}/training.log', 'a', encoding='utf-8') as log: log.write(message+'\n')
    return p

def summarize(p):
    y = imgs.label_id.values; ok = ~np.isnan(p); pred = (p >= PROB_T).astype(int)
    tp = ((pred == 1) & (y == 1) & ok).sum(); fn = ((pred == 0) & (y == 1) & ok).sum()
    fp = ((pred == 1) & (y == 0) & ok).sum(); tn = ((pred == 0) & (y == 0) & ok).sum()
    acc_k = [((pred == y) & ok & (imgs.test_fold == k)).sum() / (ok & (imgs.test_fold == k)).sum() for k in range(5)]
    return {"판정비율": round(ok.mean(), 4), "AUROC": round(roc_auc_score(y[ok], p[ok]), 4),
            "정확도": round((tp + tn) / ok.sum(), 4), "언더필 재현율": round(tp / (tp + fn), 4),
            "정상 재현율": round(tn / (tn + fp), 4), "오탐": int(fp), "미탐": int(fn),
            "fold별 정확도": " ".join(f"{a:.3f}" for a in acc_k)}

CONDS = {"A 기준선": ("nomask", False), "B 마스킹만": ("mask", False),
         "C 게이트만": ("nomask", True), "D 게이트+마스킹": ("mask", True)}
prob = {k: cv_prob(*v, condition=k) for k, v in CONDS.items()}
summary = pd.DataFrame([{"조건": k, **summarize(p)} for k, p in prob.items()])
summary.to_csv(f"{OUT}/summary.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.width", 220)
print(summary.to_string(index=False))

## 8. 원본 게이트 비율 구간별 결과
C와 D는 같은 원본 게이트 통과 사진끼리 비교한다. 재활용 목록별 집계는 아래 별도 셀에서 수행한다.

In [ ]:
y = imgs.label_id.values
bins = pd.cut(imgs.new, [0.1 - 1e-9, 0.2, 0.4, 0.7, 1.01], labels=["0.1~0.2", "0.2~0.4", "0.4~0.7", "0.7~1.0"])
rows = []
for name in ["C 게이트만", "D 게이트+마스킹"]:
    p = prob[name]; ok = ~np.isnan(p); pred = (p >= PROB_T).astype(int)
    for b in bins.cat.categories:
        s = ok & (bins == b).values
        rows.append({"조건": name, "출력물 비율": b, "정상 수": int((s & (y == 0)).sum()), "오탐": int((s & (y == 0) & (pred == 1)).sum()),
                     "언더필 수": int((s & (y == 1)).sum()), "미탐": int((s & (y == 1) & (pred == 0)).sum())})
bt = pd.DataFrame(rows).pivot(index="출력물 비율", columns="조건", values=["오탐", "미탐"])
bt.insert(0, "정상 수", pd.DataFrame(rows).groupby("출력물 비율")["정상 수"].first())
bt.insert(1, "언더필 수", pd.DataFrame(rows).groupby("출력물 비율")["언더필 수"].first())
bt.to_csv(f"{OUT}/by_ratio_bin.csv", encoding="utf-8-sig"); print(bt)

## 9. 사진별 결과 CSV + 미탐·오탐 썸네일 (D 조건)
- `predictions.csv`: 사진 한 줄씩, 4가지 조건의 확률과 결과(정답/미탐/오탐/보류). 엑셀에서 열림
- `thumbs/FN_p01.png`, `thumbs/FP_p01.png`…: D 조건에서 틀린 사진. **모델이 실제로 본 마스킹 입력**으로 보여 줍니다

In [ ]:
import matplotlib.pyplot as plt
def result(p):
    pred = (p >= PROB_T).astype(int)
    return np.where(np.isnan(p), "보류", np.where(pred == y, "정답", np.where(y == 1, "미탐", "오탐")))
df = pd.DataFrame({"사진": imgs.source, "정답": imgs.label, "시험fold": imgs.test_fold, "프레임": imgs.frame_index, "출력물비율": imgs.new})
for k, p in prob.items():
    df[f"확률_{k[0]}"] = np.round(p, 4); df[f"결과_{k[0]}"] = result(p)
df.to_csv(f"{OUT}/predictions.csv", index=False, encoding="utf-8-sig")
print(df["결과_D"].value_counts().to_dict())

TD = f"{OUT}/thumbs"; os.makedirs(TD, exist_ok=True)
pD, rD = prob["D 게이트+마스킹"], result(prob["D 게이트+마스킹"])
for kind, tag in (("미탐", "FN"), ("오탐", "FP")):
    idx = np.where(rD == kind)[0]; idx = idx[np.argsort(imgs.frame_index.values[idx])]
    for pg in range(0, len(idx), 24):
        chunk = idx[pg:pg + 24]
        fig, axs = plt.subplots(4, 6, figsize=(15, 11))
        for a in axs.flat: a.axis("off")
        for a, i in zip(axs.flat, chunk):
            mk, _ = fg.mask_bed(fg.crop_roi(cv2.imread(f"/content/dinov2_original_check/orig/{imgs.source[i]}")), bgr=True)
            a.imshow(cv2.resize(mk, (250, 250), interpolation=cv2.INTER_AREA)[:, :, ::-1])
            a.set_title(f"{imgs.label[i][:9]} #{imgs.frame_index[i]}  p={pD[i]:.2f}\nratio={imgs.new[i]:.2f}", fontsize=8)
        fig.suptitle(f"D {tag} {pg+1}-{pg+len(chunk)} / {len(idx)}")
        plt.tight_layout(); plt.savefig(f"{TD}/{tag}_p{pg//24+1:02d}.png", dpi=80)
        if pg == 0: plt.show()
        plt.close(fig)
    print(tag, len(idx), "장")

## 최종 모델 보관 — 성능 평가는 OOF만 사용
다음 셀은 원본처럼 원본 게이트 통과 사진 전체로 D 분류기를 학습해 보관한다.
**재활용 성능 집계에는 이 모델을 사용하지 않는다.**

In [ ]:
import joblib
ia = np.where(imgs.passed.values)[0]
t0 = time.time(); sc, clf = fit(X["mask"][ia], y[ia])
bundle = {"scaler": sc, "clf": clf, "backbone": "vit_base_patch14_dinov2.lvd142m", "img": IMG, "roi": fg.ROI,
          "gate_version": fg.VERSION, "gate_threshold": GATE_T, "mask": True, "prob_threshold": PROB_T, "n_train": len(ia)}
joblib.dump(bundle, f"{OUT}/fast_model_v2.joblib")
print(f"학습 {len(ia)}장, {time.time()-t0:.1f}초 → {OUT}/fast_model_v2.joblib")

## 새 이미지 판정 함수 / 속도 예시
원본 D 조건의 함수를 유지했다. 아래 기존 학습 이미지 20장 호출은 **속도 예시일 뿐, 성능평가가 아니다.**
실제 재활용 성능은 다음 셀의 OOF 결과를 확인한다.

In [ ]:
def judge(path, model=bundle):
    t0 = time.perf_counter()
    crop = fg.crop_roi(cv2.imread(str(path)))
    masked, r = fg.mask_bed(crop, bgr=True)
    prob_u = None
    if r >= model["gate_threshold"]:
        f = extract(to_input(masked[:, :, ::-1]).unsqueeze(0)).numpy()
        prob_u = float(model["clf"].predict_proba(model["scaler"].transform(f))[0, 1])
    torch.cuda.synchronize()
    verdict = "보류" if prob_u is None else ("언더필" if prob_u >= model["prob_threshold"] else "정상")
    return {"사진": Path(path).name, "출력물비율": round(r, 3), "확률": None if prob_u is None else round(prob_u, 4),
            "판정": verdict, "ms": round((time.perf_counter() - t0) * 1000, 1)}

JUDGE_DIR = None   # 예: f"{DRIVE_DIR}/new_frames"
files = sorted(glob.glob(f"{JUDGE_DIR}/*.jpg")) if JUDGE_DIR else [f"/content/dinov2_original_check/orig/{s}" for s in imgs.source.iloc[::200]]
res = pd.DataFrame([judge(f) for f in files])
res.to_csv(f"{OUT}/judged.csv", index=False, encoding="utf-8-sig")
print(res.head(10).to_string(index=False))
print(f"\n{len(res)}장 | 한 장 처리 중앙값 {res.ms.iloc[1:].median():.0f}ms (최대 {res.ms.iloc[1:].max():.0f}ms, GPU)")

## 재활용 집계와 결과 보관
런타임 반납은 아래 결과 ZIP 저장 이후 마지막 셀에서 한다.

## 재활용 사진만 따로 평가
형상 v2 마스크는 이 목록을 정하는 데만 사용했다. 모델 입력에는 **원본 DINOv2 마스크**만 사용한다.
`rescue743_all_candidates`: 총743장에서 모델이 판정한 수, 보류 수, 판정된 사진의 오탐/미탐.
`rescue_common_all4`: A~D 모두 판정한 같은 사진들로 비교. 원본 게이트가 보류한 사진을 정상 처리하지 않는다.
오탐=정상→언더필, 미탐=언더필→정상. AUROC는 확률 순위 성능이다.

In [ ]:
rows = []
for condition, p in prob.items():
    for i, r in imgs.iterrows():
        rows.append(dict(condition=condition, source=r.source, label=r.label, label_id=int(r.label_id),
                         fold=int(r.test_fold), frame_index=int(r.frame_index),
                         probability_underfill=float(p[i]) if np.isfinite(p[i]) else None,
                         pred=int(p[i]>=PROB_T) if np.isfinite(p[i]) else None,
                         pipeline_decides=int(np.isfinite(p[i]))))
oof = pd.DataFrame(rows)
oof['pred'] = oof.pred.astype('Int64')
oof.to_csv(f'{OUT}/oof_predictions_full_precision.csv', index=False, encoding='utf-8-sig')
import rescue_eval
importlib.reload(rescue_eval)
rs = pd.DataFrame(rescue_eval.evaluate(OUT, SUPPORT/'rescue_manifest.csv', SUPPORT/'source_manifest.csv', FOLDS))
print(rs[['condition', 'scope', 'total', 'judged', 'held', 'normal_held', 'underfill_held', 'fp', 'fn', 'accuracy']].to_string(index=False))
print('A→B, C→D의 같은 사진별 개선/악화:')
print(pd.read_csv(f'{OUT}/rescue_paired_changes.csv', encoding='utf-8-sig').to_string(index=False))

## 결과 공유 ZIP / 연구 한계
CSV, 설정, 로그, 확인용 그림만 공유 ZIP에 담는다. 특징과 fold 모델은 Drive에 남는다.
ZIP을 내려받아 분석 요청할 때 보내 줘. 기존 결과와 합치지 않는다.

이 결과는 클래스별 출력1회의 시간 블록 OOF이며, 새로운 출력 실험에 대한 일반화 성능이 아니다.
원본 게이트/마스킹과 회색 양이 라벨의 지름길이 될 수 있다. 재활용743장은 사용자 육안 검토를 거친
기존 선정 목록일 뿐 경계 정답 마스크가 아니다. 이 실험은 원본 모델의 해당 목록 성능 점검이며,
팀원 B 전처리와 형상 v2 조합의 재학습 효과를 증명하지 않는다.

참고: [timm DINOv2 모델](https://huggingface.co/timm/vit_base_patch14_dinov2.lvd142m),
[학습 데이터만으로 전처리 fit](https://scikit-learn.org/stable/common_pitfalls.html).

In [ ]:
result_zip = Path(DRIVE_DIR) / 'results' / f'{RUN_NAME}_results.zip'
with zipfile.ZipFile(result_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(Path(OUT).rglob('*')):
        if p.is_file() and p.suffix in {'.csv', '.json', '.log', '.png'}:
            z.write(p, p.relative_to(OUT))
print('공유할 결과 ZIP:', result_zip)
print('특징/모델은 Drive의', OUT, '에 보관됨.')

## 완료 후 런타임 반납
결과 ZIP이 Drive에 저장된 것을 확인한 뒤 아래 셀을 실행한다.

In [ ]:
from google.colab import runtime
runtime.unassign()